# HW-11: LoRA Fine-Tuning + MLflow Experiment Tracking

**Task**: Fine-tune DistilBERT for sentiment classification (SST-2) using LoRA.  
**Tracking**: Log 4 experiments with different LoRA/training configs to MLflow.  
**Dataset**: [SST-2](https://huggingface.co/datasets/glue) — binary sentiment (positive/negative), ~67k train samples.  

**Steps**:
1. Install deps & check GPU
2. Load SST-2 dataset
3. Define LoRA training function with MLflow logging
4. Run 4 experiments
5. View results in MLflow UI

> **Runtime**: Set to `T4 GPU` → Runtime → Change runtime type → T4 GPU

## 1. Install dependencies

In [ ]:
!pip install -q mlflow peft transformers datasets accelerate scikit-learn pyngrok torchao --upgrade

## 2. Check GPU

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU found. Training will be slow. Switch to T4 GPU runtime.")

## 3. Imports

In [ ]:
import os
import tempfile
from dataclasses import dataclass
from pathlib import Path
from typing import List

import mlflow
import numpy as np
from datasets import load_dataset
from peft import LoraConfig, TaskType, get_peft_model
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)
from transformers.trainer_utils import EvalPrediction

MLFLOW_EXPERIMENT = "distilbert-lora-sst2"
BASE_MODEL = "distilbert-base-uncased"
TRAIN_SUBSET = 8000   # samples from train split (keeps each run ~3-4 min on T4)
EVAL_SUBSET  = 872    # full SST-2 validation split

print("Imports OK")

## 4. Load & tokenize dataset

SST-2 from the GLUE benchmark: binary sentiment classification.  
`label=1` → positive, `label=0` → negative.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

raw = load_dataset("nyu-mll/glue", "sst2")
print(raw)

def tokenize(batch):
    return tokenizer(batch["sentence"], truncation=True, padding="max_length", max_length=128)

tokenized = raw.map(tokenize, batched=True)
tokenized = tokenized.rename_column("label", "labels")
tokenized.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

# Use a fixed subset so all experiments are comparable
train_ds = tokenized["train"].shuffle(seed=42).select(range(TRAIN_SUBSET))
eval_ds  = tokenized["validation"].select(range(EVAL_SUBSET))

print(f"Train: {len(train_ds):,}  |  Eval: {len(eval_ds):,}")
print(f"Label distribution (train): {train_ds['labels'].tolist().count(1)} pos / {train_ds['labels'].tolist().count(0)} neg")

## 5. Helper: metrics

In [ ]:
def compute_metrics(eval_pred: EvalPrediction):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy":  accuracy_score(labels, preds),
        "f1":        f1_score(labels, preds, zero_division=0),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall":    recall_score(labels, preds, zero_division=0),
    }

## 6. Training function with MLflow logging

In [ ]:
def run_experiment(
    run_name: str,
    notes: str,
    # LoRA config
    lora_rank: int = 8,
    lora_alpha: int = 16,
    lora_dropout: float = 0.1,
    target_modules: List[str] = None,
    # Training config
    epochs: int = 3,
    batch_size: int = 32,
    lr: float = 2e-5,
):
    if target_modules is None:
        target_modules = ["q_lin", "v_lin"]   # DistilBERT attention projections

    print(f"\n{'='*60}")
    print(f"  Run: {run_name}")
    print(f"  LoRA r={lora_rank} alpha={lora_alpha} dropout={lora_dropout}")
    print(f"  Training lr={lr} bs={batch_size} epochs={epochs}")
    print(f"{'='*60}")

    # Build LoRA model fresh each run
    base_model = AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL, num_labels=2
    )
    lora_cfg = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=lora_rank,
        lora_alpha=lora_alpha,
        lora_dropout=lora_dropout,
        target_modules=target_modules,
        bias="none",
    )
    model = get_peft_model(base_model, lora_cfg)
    trainable, total = model.get_nb_trainable_parameters()
    print(f"  Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

    mlflow.set_experiment(MLFLOW_EXPERIMENT)
    with mlflow.start_run(run_name=run_name):

        # --- log params ---
        mlflow.log_param("model_name",        BASE_MODEL)
        mlflow.log_param("lora_rank",         lora_rank)
        mlflow.log_param("lora_alpha",        lora_alpha)
        mlflow.log_param("lora_dropout",      lora_dropout)
        mlflow.log_param("lora_target_modules", ",".join(target_modules))
        mlflow.log_param("epochs",            epochs)
        mlflow.log_param("batch_size",        batch_size)
        mlflow.log_param("learning_rate",     lr)
        mlflow.log_param("train_samples",     len(train_ds))
        mlflow.log_param("eval_samples",      len(eval_ds))
        mlflow.log_param("trainable_params",  trainable)
        mlflow.log_param("trainable_pct",     round(100 * trainable / total, 3))
        mlflow.set_tag("notes", notes)

        with tempfile.TemporaryDirectory() as tmp:
            training_args = TrainingArguments(
                output_dir=tmp,
                eval_strategy="epoch",
                save_strategy="epoch",
                logging_strategy="steps",
                logging_steps=50,
                learning_rate=lr,
                per_device_train_batch_size=batch_size,
                per_device_eval_batch_size=batch_size,
                num_train_epochs=epochs,
                weight_decay=0.01,
                load_best_model_at_end=True,
                metric_for_best_model="f1",
                save_total_limit=1,
                report_to="none",
                fp16=torch.cuda.is_available(),
            )

            trainer = Trainer(
                model=model,
                args=training_args,
                train_dataset=train_ds,
                eval_dataset=eval_ds,
                compute_metrics=compute_metrics,
                callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
            )

            train_result = trainer.train()
            mlflow.log_metric("train_loss",       train_result.training_loss)
            mlflow.log_metric("train_runtime_s",  train_result.metrics["train_runtime"])

            final = trainer.evaluate()
            print("\nFinal eval metrics:")
            for k, v in final.items():
                print(f"  {k}: {v:.4f}")
                mlflow.log_metric(k.replace("eval_", ""), v)

            # Save & log model artifacts
            save_dir = Path(tmp) / "lora_adapter"
            trainer.save_model(str(save_dir))
            tokenizer.save_pretrained(str(save_dir))
            mlflow.log_artifacts(str(save_dir), artifact_path="lora_adapter")

    print(f"  Run '{run_name}' logged.")
    return final

## 7. Run all 4 experiments

| # | Name | LoRA rank | alpha | dropout | LR | Notes |
|---|------|-----------|-------|---------|-----|-------|
| 1 | exp1_r4_a8 | 4 | 8 | 0.1 | 2e-5 | Baseline — minimal adapter |
| 2 | exp2_r16_a32 | 16 | 32 | 0.1 | 2e-5 | Larger rank — more capacity |
| 3 | exp3_r8_lr5e-5 | 8 | 16 | 0.1 | 5e-5 | Higher LR — faster convergence |
| 4 | exp4_r8_drop0.3 | 8 | 16 | 0.3 | 2e-5 | High dropout — stronger regularization |

**Expected time**: ~3-4 min per run on T4 GPU (~15 min total)

In [ ]:
run_experiment(
    run_name="exp1_r4_a8_lr2e-5",
    notes="Baseline: rank-4 LoRA, conservative LR. Minimal trainable params (~0.08% of model).",
    lora_rank=4,
    lora_alpha=8,
    lora_dropout=0.1,
    epochs=3,
    batch_size=32,
    lr=2e-5,
)

In [ ]:
run_experiment(
    run_name="exp2_r16_a32_lr2e-5",
    notes="Larger rank (r=16). More LoRA capacity. alpha=2*r rule of thumb. Same LR as baseline.",
    lora_rank=16,
    lora_alpha=32,
    lora_dropout=0.1,
    epochs=3,
    batch_size=32,
    lr=2e-5,
)

In [ ]:
run_experiment(
    run_name="exp3_r8_a16_lr5e-5",
    notes="Medium rank (r=8), 2.5x higher LR. Tests whether faster convergence helps vs baseline.",
    lora_rank=8,
    lora_alpha=16,
    lora_dropout=0.1,
    epochs=3,
    batch_size=32,
    lr=5e-5,
)

In [ ]:
run_experiment(
    run_name="exp4_r8_a16_drop0.3",
    notes="High dropout (0.3) vs exp3. Same rank/alpha/LR. Tests regularization effect.",
    lora_rank=8,
    lora_alpha=16,
    lora_dropout=0.3,
    epochs=3,
    batch_size=32,
    lr=2e-5,
)

## 8. Compare results (inline table)

Quick summary of all runs — useful even without the MLflow UI.

In [ ]:
import pandas as pd

mlflow.set_experiment(MLFLOW_EXPERIMENT)
runs = mlflow.search_runs(order_by=["metrics.f1 DESC"])

cols = [
    "tags.mlflow.runName",
    "params.lora_rank", "params.lora_alpha", "params.lora_dropout",
    "params.learning_rate",
    "params.trainable_pct",
    "metrics.accuracy", "metrics.f1", "metrics.precision", "metrics.recall",
    "metrics.train_loss",
    "tags.notes",
]
display(runs[[c for c in cols if c in runs.columns]].rename(
    columns=lambda x: x.split(".")[-1]
).style.highlight_max(
    subset=["f1", "accuracy"], color="#d4f1c0"
).highlight_min(
    subset=["train_loss"], color="#d4f1c0"
))

## 9. Launch MLflow UI

We use **ngrok** to create a public tunnel to the MLflow server running inside Colab.

**One-time setup** (free):
1. Sign up at https://ngrok.com → Dashboard → "Your Authtoken"
2. Paste the token into the cell below

In [ ]:
NGROK_TOKEN = ""   # ← paste your ngrok authtoken here

import subprocess, time
from pyngrok import ngrok, conf

if not NGROK_TOKEN:
    print("No ngrok token provided — skipping tunnel.")
    print("MLflow data is still saved. You can download mlruns/ and run 'mlflow ui' locally.")
else:
    conf.get_default().auth_token = NGROK_TOKEN

    # Kill any existing MLflow process
    subprocess.run(["pkill", "-f", "mlflow"], capture_output=True)
    time.sleep(1)

    # Start MLflow server in background
    subprocess.Popen(
        ["mlflow", "ui", "--host", "0.0.0.0", "--port", "5000"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    time.sleep(2)

    tunnel = ngrok.connect(5000)
    print(f"\nMLflow UI: {tunnel.public_url}")
    print("Open the link above → take your screenshot for the submission")

## 10. Download mlruns/ (alternative to ngrok)

If you prefer to view the UI locally, download `mlruns/` and run `mlflow ui` on your machine.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("mlruns", "zip", ".", "mlruns")
files.download("mlruns.zip")
print("Downloaded mlruns.zip")
print("Unzip it, then run: mlflow ui")
print("Open: http://127.0.0.1:5000")